# [Thesis][Master] 3D Glaucoma — OPTIMIZED for Colab GPU

Tối ưu tốc độ train **không nén / không mất mát dữ liệu**.
Kiến trúc model & phép chuẩn hoá `x/255` giữ nguyên 100%.

**Runtime → Change runtime type → GPU** trước khi chạy.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Imports
import os, time, shutil
from glob import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

## 3. CONFIG — chỉnh ở đây

In [ ]:
DRIVE_DATA_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/dataset/"
LOCAL_DATA_DIR = "/content/dataset/"          # đĩa local Colab (rất nhanh)

COPY_TO_LOCAL        = True   # Copy 1 lần Drive -> local SSD. Lossless. Tăng tốc I/O lớn nhất.
CACHE_IN_RAM         = True   # Giữ dữ liệu (uint8) trong RAM sau lần đọc đầu. Lossless.
EVAL_TEST_EACH_EPOCH = True   # False -> chỉ chấm test ở cuối, nhanh hơn ~1/3.

USE_AMP     = True   # Mixed precision. Tăng tốc lớn trên GPU tensor core.
USE_TF32    = True   # TF32 cho Ampere+ (A100/L4). T4 không có -> tự bỏ qua.
USE_COMPILE = True   # torch.compile (PyTorch 2.x). Có overhead compile lần đầu.

BATCH_SIZE  = 4
EPOCHS      = 30
LR          = 1e-4
NUM_WORKERS = max(2, os.cpu_count() or 2)
SEED        = 42

## 4. Tối ưu backend (an toàn, miễn phí)

In [ ]:
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True          # chọn conv algo nhanh nhất cho input cố định
    if USE_TF32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

_amp_bf16  = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype  = torch.bfloat16 if _amp_bf16 else torch.float16
use_scaler = USE_AMP and device == "cuda" and amp_dtype == torch.float16
scaler     = torch.amp.GradScaler("cuda", enabled=use_scaler)

print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"AMP: {USE_AMP} (dtype={amp_dtype}, scaler={use_scaler}) | "
          f"TF32: {USE_TF32} | compile: {USE_COMPILE}")

## 5. Copy Drive → local SSD
Điểm tăng tốc I/O quan trọng nhất trên Colab. Copy nguyên bản — không mất mát.

In [ ]:
def prepare_local_data():
    if not COPY_TO_LOCAL:
        return DRIVE_DATA_DIR
    if os.path.isdir(LOCAL_DATA_DIR) and any(os.scandir(LOCAL_DATA_DIR)):
        print(f"[data] Đã có dữ liệu local tại {LOCAL_DATA_DIR}, bỏ qua copy.")
        return LOCAL_DATA_DIR

    print(f"[data] Đang copy {DRIVE_DATA_DIR} -> {LOCAL_DATA_DIR} ...")
    t0 = time.time()
    for split in ("Training", "Validation", "Test"):
        src = os.path.join(DRIVE_DATA_DIR, split)
        dst = os.path.join(LOCAL_DATA_DIR, split)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"[data] Copy xong sau {time.time() - t0:.1f}s")
    return LOCAL_DATA_DIR

BASE_DIR  = prepare_local_data()
TRAIN_DIR = os.path.join(BASE_DIR, "Training")
VAL_DIR   = os.path.join(BASE_DIR, "Validation")
TEST_DIR  = os.path.join(BASE_DIR, "Test")

## 6. Dataset
Trả về **uint8** (1,D,H,W) — `/255.0` chuyển sang GPU (kết quả y hệt). Tuỳ chọn cache RAM.

In [ ]:
class OCTDataset(Dataset):
    def __init__(self, root_dir, cache_in_ram=False):
        self.files = sorted(glob(os.path.join(root_dir, "*.pt")))
        self.cache_in_ram = cache_in_ram
        self._cache = [None] * len(self.files) if cache_in_ram else None

    def __len__(self):
        return len(self.files)

    def _load_raw(self, idx):
        try:
            sample = torch.load(self.files[idx], map_location="cpu", weights_only=True)
        except Exception:
            sample = torch.load(self.files[idx], map_location="cpu", weights_only=False)

        x = sample["oct_bscans"]                 # giữ nguyên uint8 (D,H,W)
        if x.dtype != torch.uint8:
            x = x.to(torch.uint8)
        x = x.unsqueeze(0).contiguous()          # (1,D,H,W)
        y = sample["glaucoma"].long()
        return x, y

    def __getitem__(self, idx):
        if self.cache_in_ram:
            if self._cache[idx] is None:
                self._cache[idx] = self._load_raw(idx)
            return self._cache[idx]
        return self._load_raw(idx)

## 7. Model — GIỮ NGUYÊN kiến trúc gốc

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128), nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## 8. DataLoaders

In [ ]:
_loader_workers = 0 if CACHE_IN_RAM else NUM_WORKERS
_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=_loader_workers,
    pin_memory=(device == "cuda"),
)
if _loader_workers > 0:
    _loader_kwargs.update(persistent_workers=True, prefetch_factor=4)

train_ds = OCTDataset(TRAIN_DIR, cache_in_ram=CACHE_IN_RAM)
val_ds   = OCTDataset(VAL_DIR,   cache_in_ram=CACHE_IN_RAM)
test_ds  = OCTDataset(TEST_DIR,  cache_in_ram=CACHE_IN_RAM)

train_loader = DataLoader(train_ds, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)

## 9. Helpers + Evaluate

In [ ]:
def to_device_normalize(x, y):
    x = x.to(device, non_blocking=True).float().div_(255.0)   # KẾT QUẢ Y HỆT bản gốc
    y = y.to(device, non_blocking=True)
    return x, y


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast(device_type="cuda", dtype=amp_dtype,
                            enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return accuracy_score(labels, preds)

## 10. Train

In [ ]:
model = Simple3DCNN().to(device)

if USE_COMPILE and hasattr(torch, "compile") and device == "cuda":
    try:
        model = torch.compile(model)
        print("[model] torch.compile đã bật.")
    except Exception as e:
        print(f"[model] torch.compile bỏ qua ({e}).")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_accs, test_accs = [], [], []
best_val = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    t0 = time.time()

    for x, y in train_loader:
        x, y = to_device_normalize(x, y)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype,
                            enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_acc  = evaluate(model, val_loader)
    test_acc = evaluate(model, test_loader) if EVAL_TEST_EACH_EPOCH else float("nan")

    train_losses.append(train_loss)
    val_accs.append(val_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1:03d} Loss={train_loss:.4f} "
          f"Val={val_acc:.4f} Test={test_acc:.4f} ({time.time() - t0:.1f}s)")

    if val_acc > best_val:
        best_val = val_acc
        to_save = getattr(model, "_orig_mod", model)
        torch.save(to_save.state_dict(), "best_model.pt")

if not EVAL_TEST_EACH_EPOCH:
    to_load = getattr(model, "_orig_mod", model)
    to_load.load_state_dict(torch.load("best_model.pt", weights_only=True))
    print(f"\n[final] Test accuracy (best val model) = {evaluate(model, test_loader):.4f}")

## 11. Plot

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(val_accs, label="Val")
if EVAL_TEST_EACH_EPOCH:
    plt.plot(test_accs, label="Test")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend()
plt.show()